# TheLook eCommerce: first look at the data

Goal of this notebook: get familiar with the raw tables before building anything on top of them.
Checks: table sizes, date range, which countries are in the data, state distribution for the US subset and order status breakdown.

In [11]:
import os
import sys

if os.path.exists("work/src"):
    os.chdir("work")
elif os.path.exists("../src"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from src.common.spark import get_spark
from src.ingestion.load_thelook import load_table, load_us_users
import pyspark.sql.functions as F

spark = get_spark("thelook-eda")
spark.sparkContext.setLogLevel("ERROR")

## Table sizes

In [12]:
from src.ingestion.schemas import THELOOK_SCHEMAS

for name in THELOOK_SCHEMAS:
    print(f"{name}: {load_table(spark, name).count():,} rows")

users: 100,000 rows
orders: 125,226 rows
order_items: 181,759 rows
products: 29,120 rows
events: 2,431,963 rows
distribution_centers: 10 rows
inventory_items: 490,705 rows


## Which countries are in here?

TheLook is a global dataset. The project only uses US customers because the CFPB complaint data (the NLP part) is US only, and comparing states is one of the goals.

In [13]:
users = load_table(spark, "users")
users.groupBy("country").count().orderBy(F.desc("count")).show()

+--------------+-----+
|       country|count|
+--------------+-----+
|         China|34150|
| United States|22522|
|        Brasil|14507|
|   South Korea| 5316|
|        France| 4700|
|United Kingdom| 4561|
|       Germany| 4155|
|         Spain| 4062|
|         Japan| 2438|
|     Australia| 2146|
|       Belgium| 1185|
|        Poland|  235|
|      Colombia|   17|
|        España|    2|
|   Deutschland|    2|
|       Austria|    2|
+--------------+-----+



In [14]:
us_users = load_us_users(spark)
print(f"US users: {us_users.count():,}")
us_users.groupBy("state").count().orderBy(F.desc("count")).show(15)

US users: 22,522
+--------------+-----+
|         state|count|
+--------------+-----+
|    California| 3704|
|         Texas| 2468|
|       Florida| 1698|
|      New York| 1344|
|      Illinois| 1011|
|       Georgia|  867|
|North Carolina|  829|
|       Arizona|  753|
|      Virginia|  597|
|          Ohio|  580|
|  Pennsylvania|  577|
|    Washington|  567|
|    New Jersey|  512|
|      Michigan|  497|
|      Maryland|  448|
+--------------+-----+
only showing top 15 rows


## Orders: date range and status

In [15]:
orders = load_table(spark, "orders")
orders.agg(
    F.min("created_at").alias("first_order"),
    F.max("created_at").alias("last_order"),
).show(truncate=False)

orders.groupBy("status").count().orderBy(F.desc("count")).show()

+-------------------+--------------------------+
|first_order        |last_order                |
+-------------------+--------------------------+
|2019-01-06 05:30:00|2024-01-17 19:46:14.316147|
+-------------------+--------------------------+

+----------+-----+
|    status|count|
+----------+-----+
|   Shipped|37577|
|  Complete|31354|
|Processing|25156|
| Cancelled|18609|
|  Returned|12530|
+----------+-----+



The data ends at 2024-01-17. That date acts as "today" for the whole project: recency and churn are measured against it, not against the actual calendar.

Cancelled orders (about 15%) are excluded from churn logic later on, a cancelled order is not real purchase activity.

## Null check on the columns the pipeline depends on

In [16]:
key_cols = {
    "users": ["id", "country", "state", "created_at"],
    "orders": ["order_id", "user_id", "status", "created_at"],
    "order_items": ["order_id", "user_id", "sale_price", "status"],
}
for table, cols in key_cols.items():
    df = load_table(spark, table)
    total = df.count()
    nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols])
    print(f"--- {table} ({total:,} rows), nulls per column:")
    nulls.show()

--- users (100,000 rows), nulls per column:
+---+-------+-----+----------+
| id|country|state|created_at|
+---+-------+-----+----------+
|  0|      0|    0|         0|
+---+-------+-----+----------+

--- orders (125,226 rows), nulls per column:
+--------+-------+------+----------+
|order_id|user_id|status|created_at|
+--------+-------+------+----------+
|       0|      0|     0|         0|
+--------+-------+------+----------+

--- order_items (181,759 rows), nulls per column:
+--------+-------+----------+------+
|order_id|user_id|sale_price|status|
+--------+-------+----------+------+
|       0|      0|         0|     0|
+--------+-------+----------+------+



In [17]:
spark.stop()